In [137]:
import pandas as pd 
from datetime import datetime

today_str = datetime.today().strftime("%d_%m_%Y")
raw_filename = f"data_{today_str}.csv"
clean_filename = f"data_cleaned_{today_str}.csv"


df_cleaned = pd.read_csv('/home/ji/NBA_Project/data/data_cleaned_03_03_2025.csv')
df_raw = pd.read_csv('/home/ji/NBA_Project/data/data_03_03_2025.csv')
                     
df = df_cleaned.merge(df_raw, on="GAME_ID", how="left")


In [138]:
df['HOME_AWAY'] = df['MATCHUP'].apply(lambda x: 'A' if '@' in x else 'H')

In [139]:
df_home = df[df['HOME_AWAY']=='H']
df_away = df[df['HOME_AWAY']=='A']

In [140]:
df_home['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df_home = df_home.sort_values(['TEAM_ID','GAME_DATE'])

/tmp/ipykernel_84724/3662264731.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_home['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])


In [141]:
 # List of stat columns for which to compute last 5 games average
stat_columns = [
        "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST",
        "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
    ]

In [142]:
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5OnlyHome"
    df_home[new_col] = (df_home.groupby("TEAM_ID")[col_name]
                     .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
                     .reset_index(level=0, drop=True))



In [143]:
df_home = df_home.dropna()

In [144]:
df_away['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df_away = df_away.sort_values(['TEAM_ID','GAME_DATE'])

/tmp/ipykernel_84724/3344068307.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_away['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])


In [145]:
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5Onlyaway"
    df_away[new_col] = (df_away.groupby("TEAM_ID")[col_name]
                     .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
                     .reset_index(level=0, drop=True))



In [146]:
df_away = df_away.dropna()

In [147]:
df_joined = df_away.merge(
    df_home, 
    left_on=["GAME_ID"], 
    right_on=["GAME_ID"]
    #,suffixes=('_away', '_home')
)

In [12]:
# Assuming df is your DataFrame
#df_joined.to_csv('/home/ji/NBA_Project/data/feature_engineering_home_away_stats.csv', index=False)


In [149]:
df = df_joined[df_joined['GAME_DATE']=='2025-03-02']

KeyError: 'GAME_DATE'

In [125]:
df[['TEAM_NAME_away','TEAM_NAME_home',
    'PTS_away','PTS_home','PTS_LAST5_away_away','PTS_LAST5_home_home','PTS_LAST5Onlyaway','PTS_LAST5OnlyHome']]

,TEAM_NAME_away,TEAM_NAME_home,PTS_away,PTS_home,PTS_LAST5_away_away,PTS_LAST5_home_home,PTS_LAST5Onlyaway,PTS_LAST5OnlyHome
1807,New Orleans Pelicans,Utah Jazz,128,121,111.6,112.2,110.8,112.2
2252,Chicago Bulls,Indiana Pacers,112,127,122.4,120.6,120.4,119.6
3155,Denver Nuggets,Boston Celtics,103,110,120.0,113.2,120.0,118.4
4514,LA Clippers,Los Angeles Lakers,102,108,108.4,111.4,108.4,110.6
6352,Minnesota Timberwolves,Phoenix Suns,116,98,117.4,123.8,114.2,118.6
7257,New York Knicks,Miami Heat,116,112,109.4,115.0,114.6,114.4
9534,Portland Trail Blazers,Cleveland Cavaliers,129,133,121.4,125.2,120.4,129.6
10904,Oklahoma City Thunder,San Antonio Spurs,146,132,130.4,109.0,125.0,118.6
11352,Toronto Raptors,Orlando Magic,104,102,109.0,105.0,101.6,102.6


In [132]:
 # List of stat columns for which to compute last 5 games average
stat_columns = [
        "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST",
        "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
    ]
columns =[]
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5OnlyHome"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5Onlyaway"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_away"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_home"
   # print(new_col)
    columns.append(new_col)

print(columns)
#df[df[columns]].head()

['PTS_LAST5OnlyHome', 'PTS_LAST5Onlyaway', 'PTS_LAST5_away', 'PTS_LAST5_home', 'FGM_LAST5OnlyHome', 'FGM_LAST5Onlyaway', 'FGM_LAST5_away', 'FGM_LAST5_home', 'FGA_LAST5OnlyHome', 'FGA_LAST5Onlyaway', 'FGA_LAST5_away', 'FGA_LAST5_home', 'FG_PCT_LAST5OnlyHome', 'FG_PCT_LAST5Onlyaway', 'FG_PCT_LAST5_away', 'FG_PCT_LAST5_home', 'FG3M_LAST5OnlyHome', 'FG3M_LAST5Onlyaway', 'FG3M_LAST5_away', 'FG3M_LAST5_home', 'FG3A_LAST5OnlyHome', 'FG3A_LAST5Onlyaway', 'FG3A_LAST5_away', 'FG3A_LAST5_home', 'FG3_PCT_LAST5OnlyHome', 'FG3_PCT_LAST5Onlyaway', 'FG3_PCT_LAST5_away', 'FG3_PCT_LAST5_home', 'FTM_LAST5OnlyHome', 'FTM_LAST5Onlyaway', 'FTM_LAST5_away', 'FTM_LAST5_home', 'FTA_LAST5OnlyHome', 'FTA_LAST5Onlyaway', 'FTA_LAST5_away', 'FTA_LAST5_home', 'FT_PCT_LAST5OnlyHome', 'FT_PCT_LAST5Onlyaway', 'FT_PCT_LAST5_away', 'FT_PCT_LAST5_home', 'OREB_LAST5OnlyHome', 'OREB_LAST5Onlyaway', 'OREB_LAST5_away', 'OREB_LAST5_home', 'DREB_LAST5OnlyHome', 'DREB_LAST5Onlyaway', 'DREB_LAST5_away', 'DREB_LAST5_home', 'REB_LA

In [150]:
# Get the list of column names
column_list = df.columns.tolist()

print(column_list)

['PTS_LAST5_away', 'FGM_LAST5_away', 'FGA_LAST5_away', 'FG_PCT_LAST5_away', 'FG3M_LAST5_away', 'FG3A_LAST5_away', 'FG3_PCT_LAST5_away', 'FTM_LAST5_away', 'FTA_LAST5_away', 'FT_PCT_LAST5_away', 'OREB_LAST5_away', 'DREB_LAST5_away', 'REB_LAST5_away', 'AST_LAST5_away', 'STL_LAST5_away', 'BLK_LAST5_away', 'TOV_LAST5_away', 'PF_LAST5_away', 'PLUS_MINUS_LAST5_away', 'PTS_LAST5_home', 'FGM_LAST5_home', 'FGA_LAST5_home', 'FG_PCT_LAST5_home', 'FG3M_LAST5_home', 'FG3A_LAST5_home', 'FG3_PCT_LAST5_home', 'FTM_LAST5_home', 'FTA_LAST5_home', 'FT_PCT_LAST5_home', 'OREB_LAST5_home', 'DREB_LAST5_home', 'REB_LAST5_home', 'AST_LAST5_home', 'STL_LAST5_home', 'BLK_LAST5_home', 'TOV_LAST5_home', 'PF_LAST5_home', 'PLUS_MINUS_LAST5_home', 'WL_away', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'HOME